In [ ]:
import pandas as pd
import numpy as np

# 1. Create synthetic data
np.random.seed(42)
rows = 100  # number of samples

vibration = np.random.normal(1.0, 0.2, rows)          # around 1.0 g
pressure = np.random.normal(90, 10, rows)             # 90 psi average
temperature = np.random.normal(40, 8, rows)           # 40°C avg
usage_hours = np.random.randint(0, 400, rows)         # random usage
# Maintenance more likely when vibration >1.2, temp >50, usage >300
needs_maintenance = (
    (vibration > 1.2).astype(int) +
    (temperature > 50).astype(int) +
    (usage_hours > 300).astype(int)
)
needs_maintenance = (needs_maintenance > 1).astype(int)

# 2. Combine into DataFrame
df = pd.DataFrame({
    "vibration": vibration.round(2),
    "pressure": pressure.round(1),
    "temperature": temperature.round(1),
    "usage_hours": usage_hours,
    "needs_maintenance": needs_maintenance
})

# 3. Preview data
print(df.head())


   vibration  pressure  temperature  usage_hours  needs_maintenance
0       1.10      75.8         42.9          248                  0
1       0.97      85.8         44.5          163                  0
2       1.13      86.6         48.7          393                  0
3       1.30      82.0         48.4          356                  1
4       0.95      88.4         29.0          191                  0


In [ ]:
# 1. Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 2. Use the dataset we already created
# (Assuming df is already defined from the previous step)

# 3. Define features (X) and target (y)
X = df[["vibration", "pressure", "temperature", "usage_hours"]]
y = df["needs_maintenance"]

# 4. Split into training and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Initialize Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    random_state=42,
    max_depth=5,           # controls tree size
    min_samples_split=4    # reduces overfitting
)

# 6. Train the model
rf.fit(X_train, y_train)

# 7. Predict on test data
y_pred = rf.predict(X_test)

# 8. Evaluate performance
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 9. Check feature importance
importances = pd.Series(rf.feature_importances_, index=X.columns)
print("\nFeature Importance:\n", importances.sort_values(ascending=False))



Confusion Matrix:
 [[18  1]
 [ 1  0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.95      0.95        19
           1       0.00      0.00      0.00         1

    accuracy                           0.90        20
   macro avg       0.47      0.47      0.47        20
weighted avg       0.90      0.90      0.90        20


Feature Importance:
 vibration      0.468281
temperature    0.339896
usage_hours    0.131621
pressure       0.060201
dtype: float64


In [ ]:
# 10. Display predictions alongside the actual test data

# Create a results DataFrame with test features and predictions
results = X_test.copy()
results["Actual_Needs_Maintenance"] = y_test.values
results["Predicted_Needs_Maintenance"] = y_pred

# Add a readable status column
results["Prediction_Result"] = results["Predicted_Needs_Maintenance"].map({
    1: " Maintenance Needed",
    0: " Working Fine"
})

# Reset index for cleaner display
results = results.reset_index(drop=True)

# Show the first 10 rows
print("\n Sample Predictions:\n")
print(results.head(10))



 Sample Predictions:

   vibration  pressure  temperature  usage_hours  Actual_Needs_Maintenance  \
0       0.90      94.8         30.1          169                         0   
1       1.12      92.3         48.3          378                         0   
2       1.07      81.1         51.5          234                         0   
3       0.86      97.8         37.3          338                         0   
4       0.70      92.6         30.4           51                         0   
5       1.04      77.7         46.8          264                         0   
6       1.01     104.0         45.5          360                         0   
7       0.96      96.3         40.9          173                         0   
8       0.91      70.8         44.6          151                         0   
9       1.10      75.8         42.9          248                         0   

   Predicted_Needs_Maintenance Prediction_Result  
0                            0      Working Fine  
1               

For example, even when I was simulating predictive maintenance data for EMMA, my first thought wasn’t only about accuracy — it was about how that insight would help Temple Allen reduce downtime, improve client ROI, and potentially build a new service model around it.